In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

RAW_DATA_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\data\\raw\\2025_26")
PROCESSED_DATA_DIR = Path("C:\\Users\\aabre\\Documents\\data-science-projects\\nba-player-archetype-clustering\\data\\processed\\2025_26")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_DATA_DIR

WindowsPath('C:/Users/aabre/Documents/data-science-projects/nba-player-archetype-clustering/data/processed/2025_26')

In [2]:
datasets = {}
for file in RAW_DATA_DIR.glob("*.csv"):
    dataset_name = file.stem
    if dataset_name.startswith("0"):  # Skips our inventory/summary files
        continue
    datasets[dataset_name] = pd.read_csv(file)

In [3]:
for name, df in datasets.items():
    if "PLAYER_ID" in df.columns:
        print(name, df["PLAYER_ID"].duplicated().sum())

advanced_stats 0
base_stats 0
catchshoot_stats 0
defense_stats 0
drives_stats 0
efficiency_stats 0
elbowtouch_stats 0
painttouch_stats 0
passing_stats 0
possessions_stats 0
posttouch_stats 0
pullupshot_stats 0
rebounding_stats 0
scoring_splits 0
speeddistance_stats 0
usage_stats 0


In [4]:
datasets["base_stats"]["MIN"].describe()

count     582.000000
mean     1019.758648
std       771.980982
min         2.666667
25%       293.239583
50%       940.295833
75%      1609.389167
max      2953.150000
Name: MIN, dtype: float64

In [3]:
# 250 total minutes played will be our chosen threshold for eligibility.
eligible_players = datasets["base_stats"][datasets["base_stats"]["MIN"] >= 250].copy()
eligible_players.shape

(450, 69)

In [4]:
players = eligible_players[["PLAYER_ID", "PLAYER_NAME", "TEAM_ID", "TEAM_ABBREVIATION", "AGE", "GP", "MIN"]].copy()
print(players.head())
players["PLAYER_ID"].nunique() == players.shape[0]

   PLAYER_ID    PLAYER_NAME     TEAM_ID TEAM_ABBREVIATION   AGE  GP  \
1    1631260       AJ Green  1610612749               MIL  26.0  78   
2    1642358     AJ Johnson  1610612742               DAL  21.0  48   
3     203932   Aaron Gordon  1610612743               DEN  30.0  36   
4    1628988  Aaron Holiday  1610612745               HOU  29.0  57   
5    1630174  Aaron Nesmith  1610612754               IND  26.0  45   

           MIN  
1  2269.933333  
2   453.433333  
3  1004.958333  
4   780.731667  
5  1334.950000  


True

In [6]:
player_check = []
player_ids = set(players["PLAYER_ID"])
for name, df in datasets.items():
    if "PLAYER_ID" not in df.columns:
        continue
    dataset_ids = set(df["PLAYER_ID"])
    players_available = len(player_ids.intersection(dataset_ids))
    player_check.append({
        "dataset": name, 
        "players_available": players_available, 
        "total_players": len(player_ids), 
        "coverage": (players_available / len(player_ids) * 100)
        })
player_check_df = pd.DataFrame(player_check)
player_check_df

,dataset,players_available,total_players,coverage
0,advanced_stats,450,450,100.0
1,base_stats,450,450,100.0
2,catchshoot_stats,450,450,100.0
3,defense_stats,450,450,100.0
4,drives_stats,450,450,100.0
5,efficiency_stats,450,450,100.0
6,elbowtouch_stats,450,450,100.0
7,painttouch_stats,450,450,100.0
8,passing_stats,450,450,100.0
9,possessions_stats,450,450,100.0


In [ ]:
minute_summary = {}
for name, df in datasets.items():
    if "MIN" in df.columns:
        minute_check = players[["PLAYER_ID", "PLAYER_NAME", "MIN"]].merge(
            df[["PLAYER_ID", "MIN"]], on="PLAYER_ID", how="left", suffixes=("", f"_{name}"))
        minute_check["MIN_diff"] = (minute_check["MIN"] - minute_check[f"MIN_{name}"])
        minute_summary[name] = minute_check["MIN_diff"].describe()
minute_summary_df = pd.DataFrame(minute_summary).T
minute_summary_df

KeyError: ('PLAYER_ID', 'PLAYER_NAME', 'MIN')